In [ ]:
import pandas as pd
import numpy as np

LAT_MIN, LAT_MAX = 18.3, 19.3
LON_MIN, LON_MAX = 85.32, 86.32

lat0 = (LAT_MIN + LAT_MAX) / 2
lon0 = (LON_MIN + LON_MAX) / 2


def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def process_cyclone_sheet(df, start_date, end_date):
    df["Date"] = pd.to_datetime(df["Date"])
    df["Day"] = df["Date"].dt.date

    # 9999 in data is NaN
    df["WindSpeed"] = df["WindSpeed"].replace(-9999, np.nan)

    daily = (
        df.groupby("Day")
          .agg({
              "Lat": "mean",
              "Lon": "mean",
              "WindSpeed": "max"
          })
          .reset_index()
    )

    full_days = pd.date_range(start_date, end_date, freq="D")
    fixed = pd.DataFrame({"Day": full_days.date})

    final_df = fixed.merge(daily, on="Day", how="left")

    formation_date = pd.to_datetime(start_date) + pd.Timedelta(days=10)

    # Presence = 1 indicates cyclone existence on that date
    final_df["presence"] = (
        pd.to_datetime(final_df["Day"]) >= formation_date
    ).astype(int)

    # Distance from grid center
    final_df["distance_km"] = haversine(
        final_df["Lat"], final_df["Lon"], lat0, lon0
    )

    # Cyclonic forcing
    R = 250.0  # km
    final_df["cyclonic_forcing"] = (
        final_df["WindSpeed"]
        * final_df["presence"]
        * np.exp(-final_df["distance_km"] / R)
        * 10**12
    )

    final_df["cyclonic_forcing"] = final_df["cyclonic_forcing"].fillna(0.0)

    return final_df

cyclone_windows = {
    "Phailin": ("2013-09-24", "2013-10-23"),
    "Hudhud":  ("2014-09-27", "2014-10-26"),
    "Asani":   ("2022-04-27", "2022-05-26"),
    "Titli":   ("2018-09-28", "2018-10-27"),
    "Fani":    ("2019-04-16", "2019-05-15"),
    "Amphan":  ("2020-05-06", "2020-06-04"),
    "Yaas":    ("2021-05-13", "2021-06-11"),
    "Gulab":   ("2021-09-14", "2021-10-13"),
    "Dana":    ("2024-10-12", "2024-11-10"),
}

excel_path = r"E:\ML Project\WOSC Project\EarthFormer\Cyclone_Trajectories_Data.xlsx"
xls = pd.ExcelFile(excel_path)

all_cyclones = []

for sheet in xls.sheet_names:
    if sheet not in cyclone_windows:
        print(f"Skipping sheet: {sheet}")
        continue

    print(f"Processing cyclone: {sheet}")

    start_date, end_date = cyclone_windows[sheet]
    df_sheet = pd.read_excel(xls, sheet_name=sheet)

    processed = process_cyclone_sheet(
        df_sheet, start_date, end_date
    )

    processed["Cyclone"] = sheet
    all_cyclones.append(processed)

Processing cyclone: Phailin
Processing cyclone: Hudhud
Processing cyclone: Asani
Processing cyclone: Titli
Processing cyclone: Fani
Processing cyclone: Amphan
Processing cyclone: Yaas
Processing cyclone: Gulab
Processing cyclone: Dana


In [14]:
final_dataset = pd.concat(all_cyclones, ignore_index=True)

output_path = r"E:\ML Project\WOSC Project\EarthFormer\cyclonic_forcing_by_cyclone.xlsx"

with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    for cyclone in final_dataset["Cyclone"].unique():
        final_dataset[final_dataset["Cyclone"] == cyclone] \
            .to_excel(writer, sheet_name=cyclone, index=False)

print("Saved cyclone-wise Excel file")

Saved cyclone-wise Excel file
